<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/09%20-%20Generative%20Models/09c_microscopy_diffusion_sr.ipynb)

# 09c · 材料显微图像超分辨率与去噪扩散概率模型（DDPM）入门

在 `09a` 与 `09b` 中，我们围绕一维化学序列（`SELFIES` / `SMILES`）探讨了变分自编码器（VAE）与目标导向的分子逆向设计。然而回顾 `07b`（材料显微图像 CNN），材料科学的另一大支柱是**二维与三维空间微结构（Microstructure）的表征与设计**。在这一领域，生成式深度学习正在解决两个长期困扰实验材料学家的核心痛点：

1. **低剂量/快扫电镜图像的超分辨率重建（Super-Resolution, SR）**：
   在扫描电镜（SEM）或透射电镜（TEM）表征钙钛矿薄膜、固态电解质界面（SEI）或高分子共混物时，高放大倍率、高电子束流虽然能拍出清晰的相界，但极易造成**电子束辐照损伤（Beam Damage）**且单张扫描耗时长达数分钟。如果改用低剂量、大步长快速扫描，得到的则是低分辨率（Low-Resolution, LR）且伴随散粒噪声的模糊图像。如何训练一个残差卷积网络，将模糊的低分辨率电镜图还原为相界锐利的高分辨率（High-Resolution, HR）微结构图？
2. **去噪扩散概率模型（Denoising Diffusion Probabilistic Models, DDPM）与定向微结构生成**：
   在 `09a` 中我们看到，单步解码的 VAE 在高维潜空间中面临严重的**先验空洞（Prior Hole）**——编码器很难在一步之内把复杂数据分布严丝合缝地对齐到标准正态分布 $\mathcal{N}(\mathbf{0}, \mathbf{I})$。**扩散模型（Diffusion Models；Ho et al., 2020）**彻底换了一种思路：先通过一个无需学习的**前向扩散过程（Forward Diffusion）**，分 $T$ 个微小时间步向真实图像逐步注入高斯噪声，直至第 $T$ 步严格收敛为纯粹的各向同性高斯白噪声 $\mathcal{N}(\mathbf{0}, \mathbf{I})$（从数学上根除了先验空洞！）；随后训练一个条件 U-Net 神经网络学习**逆向去噪过程（Reverse Denoising）**，从白噪声一步步雕刻出符合目标相体积分数 $\phi$ 的双相微结构。

本节我们将亲手构建这两个模型，并配备预训练好的轻量级权重 `data/micro_sr_ddpm.pt`，使所有实验在普通 CPU 上均可在数秒内流畅完成。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "09 - Generative Models"))

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

<!-- ai4m-style -->
> **课程统一绘图风格**：下方单元加载 `ai4m.mplstyle`。

In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))
import ai4m_style as ai4m

C = ai4m.use()

---
## 1. 电镜图像退化物理模型与双三次插值（Bicubic）基线

### 1.1 构建可控体积分数的双相共连续微结构数据集

在合金调幅分解（Spinodal Decomposition，Cahn-Hilliard 方程）或聚合物共混相分离中，双相微结构表现为两个互穿共连续相。我们通过高斯随机场滤波与双曲正切界面函数生成 800 张 $32 \times 32$ 的高分辨率双相显微图 $\mathbf{I}_{\text{HR}} \in [0, 1]^{1 \times 32 \times 32}$，每张图具有精确控制的亮相体积分数（在二维截面上即面积分数 $\phi \in [0.25, 0.75]$）。

同时，我们模拟低剂量快扫电镜的物理退化过程：
$$\mathbf{I}_{\text{LR}} = \text{Clamp}_{[0,1]}\!\left((K_{\text{PSF}} * \mathbf{I}_{\text{HR}}) \downarrow_{2\times} + \boldsymbol{\eta}\right), \qquad \boldsymbol{\eta} \sim \mathcal{N}(\mathbf{0}, \sigma_{\text{det}}^2 \mathbf{I})$$
即先经过 $2 \times 2$ 空间平均池化降采样至 $16 \times 16$（像素总数减少为原来的 $1/4$，对应扫描时间缩短 $75\%$），再叠加标准差 $\sigma_{\text{det}} = 0.05$ 的探测器电子噪声。

In [ ]:
def generate_two_phase_microstructures(n_samples=800, img_size=32, seed=42):
    """生成具有不同相体积分数 phi 的双相共连续微结构高分辨率图像 (N, 1, H, W)。"""
    rng = np.random.RandomState(seed)
    hr_imgs = np.zeros((n_samples, 1, img_size, img_size), dtype=np.float32)
    phis = np.zeros(n_samples, dtype=np.float32)
    for i in range(n_samples):
        raw = rng.randn(img_size, img_size)
        sigma = rng.uniform(1.8, 2.6)
        field = gaussian_filter(raw, sigma=sigma, mode="wrap")
        field = (field - field.mean()) / (field.std() + 1e-8)
        target_phi = rng.uniform(0.25, 0.75)
        thresh = np.quantile(field, 1.0 - target_phi)
        phase = 0.5 * (1.0 + np.tanh((field - thresh) / 0.18))
        hr_imgs[i, 0] = phase.astype(np.float32)
        phis[i] = float((phase > 0.5).mean())
    return torch.tensor(hr_imgs), torch.tensor(phis)


def degrade_sem_images(hr_tensor, noise_std=0.05, seed=123):
    """模拟电镜 2x 降采样快扫与探测器高斯噪声 (32x32 -> 16x16)。"""
    lr_clean = F.avg_pool2d(hr_tensor, kernel_size=2, stride=2)
    gen = torch.Generator().manual_seed(seed)
    noise = torch.randn(lr_clean.shape, generator=gen) * noise_std
    return (lr_clean + noise).clamp(0.0, 1.0)


hr_all, phi_all = generate_two_phase_microstructures(n_samples=800, img_size=32, seed=42)
lr_all = degrade_sem_images(hr_all, noise_std=0.05, seed=123)

train_hr, test_hr = hr_all[:640], hr_all[640:]
train_lr, test_lr = lr_all[:640], lr_all[640:]
train_phi, test_phi = phi_all[:640], phi_all[640:]

print(f"High-Res (HR) shape: {tuple(train_hr.shape)} | Low-Res (LR) shape: {tuple(train_lr.shape)}")

### 1.2 图像超分辨率的定量评价指标：峰值信噪比（PSNR）

在图像重建与电镜超分任务中，衡量重建图像 $\hat{\mathbf{I}} \in [0, 1]$ 与真实高分辨率图像 $\mathbf{I}_{\text{HR}} \in [0, 1]$ 接近程度的标准指标是**峰值信噪比（Peak Signal-to-Noise Ratio, PSNR，单位 dB）**：

$$\text{MSE} = \frac{1}{H W}\sum_{i=1}^{H}\sum_{j=1}^{W} \left(\hat{I}_{i,j} - I_{\text{HR}, i, j}\right)^2, \qquad \text{PSNR} = 10 \log_{10}\!\left(\frac{1^2}{\text{MSE}}\right) = -10 \log_{10}(\text{MSE})$$

由于是对数刻度，PSNR 每提升 $3\text{ dB}$ 意味着均方误差 $\text{MSE}$ 降低了一半。在传统图像处理软件中，放大低分辨率图像最常用的无学习基线是**双三次插值（Bicubic Interpolation）**——它用周围 $4 \times 4$ 邻域像素的三次多项式拟合中间像素，无法凭空恢复被低通滤波抹除的高频陡峭相界。

---
## 2. 残差超分辨率卷积网络（SR-ResCNN）：从低剂量快扫还原清晰相界

为了恢复陡峭的两相界面并滤除探测器散粒噪声，我们构建一个轻量级**残差超分辨率卷积网络（`MicrostructureSRNet`）**。其核心架构设计遵循两条物理直觉：

1. **先插值对齐空间网格**：先将 $16 \times 16$ 的低分辨率输入 $\mathbf{I}_{\text{LR}}$ 通过双三次插值放大到 $32 \times 32$ 的初猜场 $\mathbf{I}_{\text{bicubic}}$；
2. **残差高频学习（Residual Learning）**：由于 $\mathbf{I}_{\text{bicubic}}$ 已经包含了正确的低频相分布轮廓，卷积主干网络无需从零画起，只需专门预测**高频相界修正残差** $\Delta \mathbf{I} = f_\theta(\mathbf{I}_{\text{bicubic}})$，最终输出 $\hat{\mathbf{I}}_{\text{HR}} = \text{Clamp}_{[0,1]}(\mathbf{I}_{\text{bicubic}} + \Delta \mathbf{I})$。

In [ ]:
class MicrostructureSRNet(nn.Module):
    """残差超分辨率卷积网络：在双三次插值初猜场上学习高频相界残差。"""
    def __init__(self, channels=32):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, 1, kernel_size=3, padding=1),
        )

    def forward(self, x_lr):
        x_up = F.interpolate(x_lr, scale_factor=2, mode="bicubic", align_corners=False)
        res = self.body(x_up)
        return (x_up + res).clamp(0.0, 1.0)


DATA_DIR = "data" if os.path.exists("data/micro_sr_ddpm.pt") else "course/09 - Generative Models/data"
ckpt_path = os.path.join(DATA_DIR, "micro_sr_ddpm.pt")
TRAIN_SR_FROM_SCRATCH = False

sr_net = MicrostructureSRNet(channels=32).to(device)
if os.path.exists(ckpt_path) and not TRAIN_SR_FROM_SCRATCH:
    ckpt_09c = torch.load(ckpt_path, map_location=device)
    sr_net.load_state_dict(ckpt_09c["sr_state_dict"])
    sr_history = ckpt_09c["sr_history"]
    print(f"Loaded pretrained MicrostructureSRNet from {ckpt_path}")
else:
    opt_sr = torch.optim.Adam(sr_net.parameters(), lr=2e-3)
    sr_history = []
    for ep in range(1, 26):
        sr_net.train()
        perm_sr = torch.randperm(len(train_hr))
        ep_loss = 0.0
        for i in range(0, len(train_hr), 64):
            idx = perm_sr[i : i + 64]
            bx, by = train_lr[idx].to(device), train_hr[idx].to(device)
            loss = F.mse_loss(sr_net(bx), by)
            opt_sr.zero_grad()
            loss.backward()
            opt_sr.step()
            ep_loss += loss.item() * len(idx)
        sr_history.append(ep_loss / len(train_hr))
sr_net.eval();

In [ ]:
# 对比测试集上双三次插值 (Bicubic) 与 SR-ResCNN 的 PSNR 及体视学面积分数误差
with torch.no_grad():
    bic_test = F.interpolate(test_lr, scale_factor=2, mode="bicubic", align_corners=False).clamp(0.0, 1.0)
    sr_test = sr_net(test_lr.to(device)).cpu()

def calc_psnr_batch(pred, target):
    mse_per_img = ((pred - target) ** 2).mean(dim=(1, 2, 3))
    return (-10.0 * torch.log10(mse_per_img + 1e-10)).mean().item()

psnr_bic = calc_psnr_batch(bic_test, test_hr)
psnr_sr = calc_psnr_batch(sr_test, test_hr)

# 测量二值化阈值分割后的亮相面积分数绝对误差 (MAE of Phase Fraction phi)
phi_true = (test_hr > 0.5).float().mean(dim=(1, 2, 3))
phi_bic = (bic_test > 0.5).float().mean(dim=(1, 2, 3))
phi_sr = (sr_test > 0.5).float().mean(dim=(1, 2, 3))

mae_phi_bic = (phi_bic - phi_true).abs().mean().item() * 100.0
mae_phi_sr = (phi_sr - phi_true).abs().mean().item() * 100.0

print(f"Test Set PSNR               | Bicubic: {psnr_bic:.2f} dB  -->  SR-ResCNN: {psnr_sr:.2f} dB (+{psnr_sr - psnr_bic:.2f} dB)")
print(f"Phase Fraction MAE (% area) | Bicubic: {mae_phi_bic:.2f}%    -->  SR-ResCNN: {mae_phi_sr:.2f}%")

In [ ]:
# 可视化对比：低分辨率输入、双三次插值、SR-ResCNN 重建与真实高分辨率图像 + 残差场
sample_id = 3
fig, axes = plt.subplots(1, 5, figsize=(12.5, 2.8))
panels = [
    (test_lr[sample_id, 0], "16x16 Noisy LR Input", "gray"),
    (bic_test[sample_id, 0], f"Bicubic ({psnr_bic:.1f} dB)", "gray"),
    (sr_test[sample_id, 0], f"SR-ResCNN ({psnr_sr:.1f} dB)", "gray"),
    (test_hr[sample_id, 0], "32x32 Ground Truth HR", "gray"),
    ((sr_test[sample_id, 0] - bic_test[sample_id, 0]), "Learned Edge Residual", "RdBu_r"),
]
for ax, (img, title, cmap_name) in zip(axes, panels):
    ax.imshow(img.numpy(), cmap=cmap_name)
    ax.set_title(title, fontsize=9.5)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

观察上方最右侧的第 5 张图（`Learned Edge Residual`，即网络学到的残差场 $\Delta \mathbf{I} = \hat{\mathbf{I}}_{\text{SR}} - \mathbf{I}_{\text{bicubic}}$）：网络自动在相界内侧涂上正残差（红色加亮），在相界外侧涂上负残差（蓝色压暗），同时在平坦相内部抵消了散粒噪声，从而将测试集平均 PSNR 提升了超过 **5 dB**，并将下游体视学相面积分数的测量误差削减了一半以上！

### 练习 09c-1：构建 Sobel 边缘梯度损失（Edge Gradient Loss）评估相界清晰度

仅用像素均方误差（MSE）训练超分网络时，模型为了最小化平均平方误差，往往倾向于在相界处取折中的模糊灰度值。在材料金相超分中，研究者常在损失函数中额外加入利用 `07b` 讲过的 **Sobel 算子**提取的**一阶空间梯度误差**，强制模型还原锐利的相边界：

$$\mathcal{L}_{\text{Sobel}}(\hat{\mathbf{I}}, \mathbf{I}) = \|\mathbf{K}_x * \hat{\mathbf{I}} - \mathbf{K}_x * \mathbf{I}\|_1 + \|\mathbf{K}_y * \hat{\mathbf{I}} - \mathbf{K}_y * \mathbf{I}\|_1$$

请在下方实现函数 `sobel_gradient_loss(pred, target)`，并用它对比 `bic_test` 与 `sr_test` 在测试集上的相界梯度误差（验证 `sr_test` 的相界梯度误差显著低于双三次插值）。

In [ ]:
# 练习 09c-1：在此使用 F.conv2d 实现 Sobel 梯度损失并对比 bic_test 与 sr_test
def sobel_gradient_loss(pred, target):
    # TODO:
    # 1. 定义 3x3 水平与垂直 Sobel 卷积核 K_x, K_y
    # 2. 用 F.conv2d(..., padding=1) 提取 pred 与 target 的水平和垂直梯度图
    # 3. 返回 L1 梯度损失均值
    pass


---
## 3. 去噪扩散概率模型（DDPM）核心数学：前向加噪与逆向去噪

在前半节中，超分辨率网络解决的是“一对一映射”的图像恢复问题。现在我们转向真正的**从零生成新微结构**：如何训练一个生成模型，不仅不存在 `09a` 变分自编码器的“先验空洞”，还能按照工程师指定的目标相体积分数 $\phi_{\text{target}}$ 合成全新的二维双相组织？

### 3.1 前向扩散过程（Forward Diffusion Process）与一步闭合跳步公式

设真实微结构图像（线性缩放到 $[-1, 1]$ 区间）为 $\mathbf{x}_0$。**前向扩散过程**是一个固定的、不含任何可学习参数的马尔可夫链：在时间步 $t = 1, 2, \dots, T$，按预设的微小方差调度 $\beta_1 < \beta_2 < \dots < \beta_T \in (0, 1)$ 逐步混入标准高斯白噪声：

$$q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{1 - \beta_t}\,\mathbf{x}_{t-1},\; \beta_t \mathbf{I}\right)$$

记 $\alpha_t = 1 - \beta_t$，以及累积信号保留率 $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$。利用高斯分布独立可加的性质（与 `09a` 的重参数化技巧同源），我们**不需要在训练时真的循环迭代 $t$ 步**，而是可以用一个极其优美的闭合公式，从干净图像 $\mathbf{x}_0$ 一步直接跳到任意噪声时刻 $t$：

$$q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1 - \bar{\alpha}_t)\mathbf{I}\right) \quad \Longleftrightarrow \quad \mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\,\boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

当步数 $T$ 足够且 $\bar{\alpha}_T \to 0$ 时，终点状态 $\mathbf{x}_T \approx \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ 精确退化为纯粹的标准正态分布。这就从数学根源上解释了**为什么扩散模型没有 VAE 的先验空洞**：VAE 靠一个容量有限的编码器一步硬把数据塞进 $\mathcal{N}(\mathbf{0}, \mathbf{I})$ 往往塞不满，而扩散模型的前向过程用解析高斯混合直接把数据精确溶解成了标准正态白噪声！

In [ ]:
# 定义 T=40 步的线性噪声调度，并可视化一张双相微结构在前向扩散过程中的渐进溶解
x0_16 = F.avg_pool2d(hr_all, kernel_size=2, stride=2) * 2.0 - 1.0  # (800, 1, 16, 16) in [-1, 1]

T_STEPS = 40
betas = torch.linspace(1e-4, 0.08, T_STEPS)
alphas = 1.0 - betas
alphas_bar = torch.cumprod(alphas, dim=0)

def q_sample(x0, t_idx, noise=None):
    """前向扩散闭合公式：一步生成时刻 t_idx 的加噪图像 x_t。"""
    if noise is None:
        noise = torch.randn_like(x0)
    ab_t = alphas_bar.to(x0.device)[t_idx][:, None, None, None]
    return torch.sqrt(ab_t) * x0 + torch.sqrt(1.0 - ab_t) * noise, noise

demo_x0 = x0_16[0:1]
show_steps = [0, 5, 12, 20, 30, 39]
fig, axes = plt.subplots(1, len(show_steps), figsize=(12, 2.3))
torch.manual_seed(0)
for ax, step in zip(axes, show_steps):
    if step == 0:
        xt_vis = demo_x0[0, 0]
    else:
        xt_vis, _ = q_sample(demo_x0, torch.tensor([step]))
        xt_vis = xt_vis[0, 0]
    ax.imshow(xt_vis.numpy(), cmap="gray", vmin=-1.2, vmax=1.2)
    ax.set_title(f"t = {step} (ᾱ_t={alphas_bar[step]:.2f})", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("Forward Diffusion Process q(x_t | x_0): Microstructure Dissolving into Gaussian Noise", y=1.03)
plt.tight_layout()
plt.show()

### 练习 09c-2：验证前向扩散信噪比（SNR）衰减与终端先验对齐

定义时刻 $t$ 的**信噪比（Signal-to-Noise Ratio, SNR）**为信号方差权重与噪声方差权重之比：

$$\text{SNR}(t) = \frac{\bar{\alpha}_t}{1 - \bar{\alpha}_t}$$

请在下方编写代码：
1. 计算 $t = 0, 1, \dots, T-1$ 的 $\text{SNR}(t)$（以分贝 $10\log_{10}\text{SNR}(t)$ 表示）；
2. 对全部 800 张真实微结构 `x0_16` 在终点 $t = T - 1$ 执行 `q_sample`，计算生成的 `x_T` 的像素均值、标准差以及模长中位数 $\|\mathbf{x}_T\|_2$，并与纯高斯白噪声 `torch.randn_like(x0_16)` 的模长中位数对比，验证两者高度重合（无先验空洞）。

In [ ]:
# 练习 09c-2：在此计算 SNR(t) 衰减并对比 x_T 与 N(0,I) 的模长分布
# TODO:
# 1. 计算 snr_db = 10 * torch.log10(alphas_bar / (1 - alphas_bar))
# 2. 对 x0_16 在 t = T_STEPS - 1 采样 x_T，对比 x_T.flatten(1).norm(dim=1).median() 与标准正态白噪声


---
## 4. 条件扩散模型（Conditional DDPM）：按目标体积分数定制双相微结构

### 4.1 噪声预测网络 `ConditionalMicroUNet` 与简化训练目标

既然前向过程是不断加噪，那么生成新微结构的**逆向过程（Reverse Process）**就是从纯白噪声 $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ 出发，按 $t = T-1, T-2, \dots, 0$ 逐步减去噪声。Ho 等人（2020）证明：与其让神经网络直接预测去噪后的图像，不如让一个以当前加噪图 $\mathbf{x}_t$、时间步 $t$ 以及条件变量 $\phi$（目标亮相体积分数）为输入的 U-Net 网络 $\hat{\boldsymbol{\epsilon}}_\theta(\mathbf{x}_t, t, \phi)$，直接预测**当初混入的那份标准正态噪声 $\boldsymbol{\epsilon}$**！

训练目标函数由此简化为极其稳定的均方误差（Simple Noise-Prediction MSE）：

$$\mathcal{L}_{\text{DDPM}}(\theta) = \mathbb{E}_{\mathbf{x}_0,\, t,\, \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0},\mathbf{I})}\!\left[\left\|\boldsymbol{\epsilon} - \hat{\boldsymbol{\epsilon}}_\theta\!\left(\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\,\boldsymbol{\epsilon},\; t,\; \phi\right)\right\|_2^2\right]$$

一旦训练好噪声预测器 $\hat{\boldsymbol{\epsilon}}_\theta$，在推理采样时，每一步从 $\mathbf{x}_t$ 退火到 $\mathbf{x}_{t-1}$ 的逆向更新公式为：

$$\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\,\hat{\boldsymbol{\epsilon}}_\theta(\mathbf{x}_t, t, \phi)\right) + \sqrt{\beta_t}\,\mathbf{z}, \qquad \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}) \; (t > 0)$$

下面我们定义带跳跃连接（Skip Connection）与 $(t, \phi)$ 条件注入的轻量级 U-Net `ConditionalMicroUNet`，并加载预训练权重。

In [ ]:
class ConditionalMicroUNet(nn.Module):
    """以扩散步 t 和目标相体积分数 phi 为条件的轻量级微结构去噪 U-Net。"""
    def __init__(self, base_ch=32, T=40):
        super().__init__()
        self.T = T
        self.cond_mlp = nn.Sequential(
            nn.Linear(2, base_ch),
            nn.SiLU(),
            nn.Linear(base_ch, base_ch),
        )
        self.enc1 = nn.Sequential(nn.Conv2d(1, base_ch, 3, padding=1), nn.GroupNorm(4, base_ch), nn.SiLU())
        self.down = nn.Conv2d(base_ch, base_ch * 2, 4, stride=2, padding=1)
        self.mid = nn.Sequential(
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, padding=1),
            nn.GroupNorm(8, base_ch * 2),
            nn.SiLU(),
            nn.Conv2d(base_ch * 2, base_ch * 2, 3, padding=1),
            nn.GroupNorm(8, base_ch * 2),
            nn.SiLU(),
        )
        self.up = nn.ConvTranspose2d(base_ch * 2, base_ch, 4, stride=2, padding=1)
        self.out_conv = nn.Sequential(
            nn.Conv2d(base_ch * 2, base_ch, 3, padding=1),
            nn.GroupNorm(4, base_ch),
            nn.SiLU(),
            nn.Conv2d(base_ch, 1, 3, padding=1),
        )

    def forward(self, x_t, t, phi):
        t_norm = t.float().unsqueeze(1) / float(self.T)
        phi_col = phi.float().unsqueeze(1)
        cond = self.cond_mlp(torch.cat([t_norm, phi_col], dim=1))[:, :, None, None]

        h1 = self.enc1(x_t) + cond
        h2 = F.silu(self.down(h1))
        h_mid = self.mid(h2)
        h_up = self.up(h_mid)
        return self.out_conv(torch.cat([h_up, h1], dim=1))


ddpm_net = ConditionalMicroUNet(base_ch=32, T=T_STEPS).to(device)
TRAIN_DDPM_FROM_SCRATCH = False

if os.path.exists(ckpt_path) and not TRAIN_DDPM_FROM_SCRATCH:
    ddpm_net.load_state_dict(ckpt_09c["ddpm_state_dict"])
    ddpm_history = ckpt_09c["ddpm_history"]
    print(f"Loaded pretrained ConditionalMicroUNet from {ckpt_path}")
else:
    opt_ddpm = torch.optim.AdamW(ddpm_net.parameters(), lr=3e-3, weight_decay=1e-4)
    x0_dev, phi_dev, ab_dev = x0_16.to(device), phi_all.to(device), alphas_bar.to(device)
    ddpm_history = []
    for ep in range(1, 61):
        ddpm_net.train()
        perm_d = torch.randperm(len(x0_dev), device=device)
        ep_loss = 0.0
        for i in range(0, len(x0_dev), 64):
            idx = perm_d[i : i + 64]
            bx0, bphi = x0_dev[idx], phi_dev[idx]
            bt = torch.randint(0, T_STEPS, (len(idx),), device=device)
            noise = torch.randn_like(bx0)
            ab_t = ab_dev[bt][:, None, None, None]
            x_t = torch.sqrt(ab_t) * bx0 + torch.sqrt(1.0 - ab_t) * noise
            loss = F.mse_loss(ddpm_net(x_t, bt, bphi), noise)
            opt_ddpm.zero_grad()
            loss.backward()
            opt_ddpm.step()
            ep_loss += loss.item() * len(idx)
        ddpm_history.append(ep_loss / len(x0_dev))
ddpm_net.eval();

### 4.2 逆向去噪采样与相组织涌现过程可视化

下面我们实现完整的逆向采样函数 `sample_ddpm_microstructures(model, target_phis)`：从纯随机高斯白噪声 $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ 出发，观察模型如何在 40 个时间步内一步步抹去噪声、凝聚相界，并同时按给定目标体积分数 $\phi_{\text{target}} \in \{0.30, 0.50, 0.70\}$ 定制出不同相比例的微结构（最后还可串联第 2 节的 `sr_net` 直接放大为 $32 \times 32$ 高清微结构！）。

In [ ]:
@torch.no_grad()
def sample_ddpm_microstructures(model, target_phis, record_trajectory=False, seed=7):
    """按给定目标相体积分数向量 target_phis 执行 DDPM 逆向采样。"""
    model.eval()
    torch.manual_seed(seed)
    n = len(target_phis)
    phi_t = torch.tensor(target_phis, dtype=torch.float32, device=device)
    x = torch.randn(n, 1, 16, 16, device=device)

    traj = {}
    for step in reversed(range(T_STEPS)):
        t_batch = torch.full((n,), step, dtype=torch.long, device=device)
        pred_noise = model(x, t_batch, phi_t)
        alpha_t = alphas[step].to(device)
        ab_t = alphas_bar[step].to(device)
        beta_t = betas[step].to(device)

        mean = (1.0 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1.0 - ab_t)) * pred_noise)
        if step > 0:
            z = torch.randn_like(x)
            x = mean + torch.sqrt(beta_t) * z
        else:
            x = mean
        if record_trajectory and step in (39, 28, 18, 10, 4, 0):
            traj[step] = ((x.clamp(-1.0, 1.0) + 1.0) * 0.5).cpu()

    imgs_01 = ((x.clamp(-1.0, 1.0) + 1.0) * 0.5).cpu()
    return (imgs_01, traj) if record_trajectory else imgs_01


# 1. 展示单个微结构从 t=39 (白噪声) 到 t=0 (清晰双相结构) 的逆向涌现轨迹
_, traj_dict = sample_ddpm_microstructures(ddpm_net, [0.50], record_trajectory=True, seed=11)
traj_steps = [39, 28, 18, 10, 4, 0]

fig, axes = plt.subplots(1, len(traj_steps), figsize=(12, 2.3))
for ax, st in zip(axes, traj_steps):
    ax.imshow(traj_dict[st][0, 0].numpy(), cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_title(f"Reverse Step t = {st}", fontsize=9.5)
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("Reverse Denoising Trajectory p_θ(x_{t-1} | x_t): Phase Morphology Emerging from Pure Noise", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# 2. 按不同目标体积分数 phi_target = 0.30, 0.50, 0.70 定向生成微结构，并用 SR-ResCNN 超分至 32x32
target_list = [0.30, 0.30, 0.50, 0.50, 0.70, 0.70]
gen_16 = sample_ddpm_microstructures(ddpm_net, target_list, seed=21)
with torch.no_grad():
    gen_32_sr = sr_net(gen_16.to(device)).cpu()

fig, axes = plt.subplots(1, 6, figsize=(12.5, 2.5))
for i, ax in enumerate(axes):
    measured_phi = float((gen_32_sr[i, 0] > 0.5).float().mean().item())
    ax.imshow(gen_32_sr[i, 0].numpy(), cmap="gray", vmin=0.0, vmax=1.0)
    ax.set_title(f"Target φ={target_list[i]:.2f}\nActual φ̂={measured_phi:.2f}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("Conditional DDPM + SR-ResCNN: Goal-Directed Microstructure Synthesis", y=1.05)
plt.tight_layout()
plt.show()

### 练习 09c-3：条件扩散微结构生成的体视学校准曲线（Target $\phi$ vs. Measured $\hat{\phi}$）

为了定量验证条件扩散模型是否真正听懂了输入的连续条件参数 $\phi_{\text{target}}$，请在下方编写代码：
1. 对四个目标体积分数档位 $\phi_{\text{target}} \in \{0.30, 0.45, 0.55, 0.70\}$，每个档位各条件生成 20 张微结构图像（共 80 张）；
2. 以 `0.5` 为阈值对生成图像二值化，统计每张图的实际亮相面积分数 $\hat{\phi} = \text{mean}(I > 0.5)$；
3. 计算每个档位下实测 $\hat{\phi}$ 的均值与标准差，并绘制 $\phi_{\text{target}}$ vs. $\hat{\phi}$ 校准对比图。

In [ ]:
# 练习 09c-3：在此批量生成不同目标体积分数的微结构并验证实测面积分数校准度
# TODO: 对 [0.30, 0.45, 0.55, 0.70] 每个档位生成 20 张微结构，计算实测面积分数均值与标准差


---
## 5. 本节小结与模块全景对比

至此，我们完成了第 08 讲与第 09 讲涵盖的全部三大类主流生成模型。将它们的核心机制与材料科学适用场景总结如下：

| 生成模型家族 | 核心数学机制 | 优势 | 局限与防坑要点 | 典型材料/化学应用 |
| :--- | :--- | :--- | :--- | :--- |
| **自回归序列模型**<br>（LSTM / Transformer，`08a`–`08c`） | 链式法则分解 $P(x_1, \dots, x_T) = \prod_t P(x_t \mid x_{<t})$ | 序列分布保真度高，天然契合变长 SMILES 与反应翻译 | 无固定维度全局连续潜坐标；长序列逐 token 解码较慢 | 有机小分子从头生成、小样本迁移微调、正向/逆合成反应预测 |
| **变分自编码器**<br>（$\beta$-VAE，`09a`） | 编码对角高斯后验 $q_\phi(\mathbf{z}\mid\mathbf{x})$ + 重参数化 + KL 先验正则 | 提供显式固定维度连续潜向量 $\mathbf{z}$，支持球面插值（Slerp）与局部邻域搜索 | 一步压缩易产生**先验空洞（Prior Hole）**；全局先验采样易偏离训练流形 | 分子潜空间插值、种子分子局部结构演化 |
| **去噪扩散概率模型**<br>（DDPM，`09c`） | 前向解析加噪至纯高斯白噪声 $\mathcal{N}(\mathbf{0}, \mathbf{I})$ + 逆向迭代预测噪声 $\hat{\boldsymbol{\epsilon}}_\theta(\mathbf{x}_t, t, c)$ | **彻底消除先验空洞**，条件控制精准，三维/二维空间结构生成质量极高 | 逆向采样需要迭代数十至数百步 U-Net/GNN 前向传播 | 电镜微结构生成与修复、三维晶体结构生成（DiffCSP / MatterGen）与三维分子构象生成 |

在下一讲（也是本课程的压轴模块 **10 · 图神经网络与机器学习原子间势函数**）中，我们将彻底告别把分子当作字符串或二维像素网格的间接表示，直接将分子与晶体表示为**三维空间中的图（Graphs：节点 = 原子，边 = 化学键/邻域距离）**，亲手构建消息传递图神经网络（MPNN / Chemprop D-MPNN）、晶体图卷积网络（CGCNN）以及驱动现代原子尺度分子动力学模拟的等变机器学习势函数（MACE）。